# speech playground

record your voice -> transcribe (stt), design a voice -> generate speech (tts).
reads `RUNPOD_API_KEY`, `STT_ENDPOINT_ID`, `TTS_ENDPOINT_ID` from `.env`.

In [2]:
import base64
import io
import os
from pathlib import Path

import requests
from IPython.display import Audio, display

env_path = next(
    (p / ".env" for p in [Path.cwd(), *Path.cwd().parents] if (p / ".env").exists()), None
)
for line in env_path.read_text().splitlines():
    key, _, value = line.partition("=")
    os.environ.setdefault(key.strip(), value.strip())

API = "https://api.runpod.ai/v2"
HEADERS = {"Authorization": f"Bearer {os.environ['RUNPOD_API_KEY']}"}


def call(endpoint_id, input_payload):
    resp = requests.post(
        f"{API}/{endpoint_id}/runsync",
        headers=HEADERS,
        json={"input": input_payload},
        timeout=600,
    )
    resp.raise_for_status()
    output = resp.json().get("output") or {}
    if output.get("error"):
        raise RuntimeError(output["error"])
    return output


def transcribe(audio_bytes, language=None):
    payload = {"audio": base64.b64encode(audio_bytes).decode()}
    if language:
        payload["language"] = language
    return call(os.environ["STT_ENDPOINT_ID"], payload)


def speak(text, instruct="", language="Auto", **sampling):
    output = call(
        os.environ["TTS_ENDPOINT_ID"],
        {"text": text, "instruct": instruct, "language": language, **sampling},
    )
    audio = base64.b64decode(output["audio"])
    return Audio(audio, rate=output["sample_rate"])


def chat(messages, max_tokens=256, **sampling):
    output = call(
        os.environ["LLM_ENDPOINT_ID"],
        {"messages": messages, "max_tokens": max_tokens, **sampling},
    )
    print(output["text"])
    return output


def record(seconds=5, rate=16000):
    import sounddevice as sd
    import soundfile as sf

    print(f"recording {seconds}s... speak now")
    audio = sd.rec(int(seconds * rate), samplerate=rate, channels=1, dtype="int16")
    sd.wait()
    buffer = io.BytesIO()
    sf.write(buffer, audio, rate, format="WAV")
    return buffer.getvalue()


print("ready")

ready


## 1. record your voice and transcribe it

In [3]:
my_voice = record(seconds=5)
display(Audio(my_voice, rate=16000))
transcribe(my_voice)

recording 5s... speak now


{}

## 2. design a voice and listen

In [ ]:
speak(
    "Hi! This voice was designed on demand, just for you.",
    instruct="A cute child's voice, around 8 years old, speaking with a "
    "slightly childish tone, suitable for animation character voice-overs.",
    language="English",
)

b'RIFF$\xfd\x02\x00WAVEfmt \x10\x00\x00\x00\x01\x00\x01\x00\xc0]\x00\x00\x80\xbb\x00\x00\x02\x00\x10\x00data\x00\xfd\x02\x00\x00\x00\x01\x00\x01\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\x00\x01\x00\x00\x00\xff\xff\xfe\xff\xfe\xff\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\xff\xff\xff\xff\x00\x00\x00\x00\x00\x00\xff\xff\xff\xff\x00\x00\x01\x00\xff\xff\x00\x00\x00\x00\x00\x00\x00\x00\xfe\xff\xff\xff\x00\x00\x00\x00\x00\x00\x01\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\xff\xff\xff\xff\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\xff\xff\xff\xff\xff\xff\x00\x00\x01\x00\x00\x00\xff\xff\xfe\xff\xff\xff\x00\x00\x00\x00\xff\xff\xff\xff\xff\xff\x00\x00\x00\x00\xff\xff\xff\xff\xff\xff\xff\xff\x00\x00\x00\x00\x00\x00\xff\xff\xff\xff\x00\x00\x00\x00\xff\xff\xfe\xff\xff\xff\xfe\xff\xff\xff\x00\x00\xff\xff\xff\xff\xfe\xff\xff\xff\xff\xff\x00\x00\x00\x00\x00\x00\x00\x00\x01\x00\x02

In [5]:
# another voice - same text, different character
speak(
    "Hi! This voice was designed on demand, just for you.",
    instruct="elderly man, gravelly voice, slow and thoughtful",
    language="English",
    temperature=0.8,
)

b'RIFF$G\x04\x00WAVEfmt \x10\x00\x00\x00\x01\x00\x01\x00\xc0]\x00\x00\x80\xbb\x00\x00\x02\x00\x10\x00data\x00G\x04\x00\x04\x00\x06\x00\x06\x00\x06\x00\x06\x00\x06\x00\x05\x00\x06\x00\x05\x00\x03\x00\x03\x00\x05\x00\x06\x00\x05\x00\x05\x00\x06\x00\x06\x00\x04\x00\x04\x00\x04\x00\x03\x00\x05\x00\x06\x00\x06\x00\x06\x00\x06\x00\x06\x00\x07\x00\x08\x00\x07\x00\x07\x00\x07\x00\x08\x00\t\x00\t\x00\n\x00\x0b\x00\x0b\x00\t\x00\n\x00\t\x00\n\x00\n\x00\x0b\x00\x0b\x00\x0c\x00\x0c\x00\r\x00\x0e\x00\x0f\x00\x0f\x00\x0f\x00\x10\x00\x10\x00\x0f\x00\r\x00\x0e\x00\x0e\x00\x0e\x00\x0e\x00\x0c\x00\x0f\x00\x0f\x00\x10\x00\x11\x00\x11\x00\x12\x00\x14\x00\x13\x00\x12\x00\x13\x00\x14\x00\x13\x00\x12\x00\x11\x00\x11\x00\x11\x00\x11\x00\x11\x00\x11\x00\x12\x00\x13\x00\x13\x00\x13\x00\x13\x00\x12\x00\x13\x00\x12\x00\x12\x00\x12\x00\x12\x00\x13\x00\x13\x00\x14\x00\x14\x00\x15\x00\x15\x00\x16\x00\x16\x00\x15\x00\x14\x00\x14\x00\x14\x00\x14\x00\x13\x00\x13\x00\x15\x00\x14\x00\x14\x00\x16\x00\x17\x00\x15\x00\x17\x

## 4. chat with qwen3-14b


In [ ]:
chat([{"role": "user", "content": "Explain speculative decoding in two sentences."}])


## 5. full loop: llm writes it, tts speaks it


In [ ]:
reply = chat(
    [{"role": "user", "content": "Say something inspiring in one short sentence."}],
    max_tokens=64,
)
speak(reply["text"], instruct="warm confident narrator, medium pace", language="English")
